# NN-Steiner on Colab

Run the cells top to bottom at the start of every session.

- **Use a GPU runtime** for training: *Runtime → Change runtime type → T4 GPU*.
- Datasets, checkpoints, and results are saved to the shared Drive folder below, so they survive when Colab resets.
- The code comes from GitHub. To change it, edit it locally, push, and re-run step 2 here.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Shared team folder. Everything our runs generate is kept here.
DRIVE_WORK = '/content/drive/MyDrive/FA26CMPE257Sec02MachineLearningProject/nn-steiner-work'

## 2. Get the latest code

In [ ]:
import os

REPO = '/content/nn-steiner-rsmt'
if not os.path.exists(REPO):
    !git clone https://github.com/Ashilman25/nn-steiner-rsmt.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}

## 3. Keep generated files on Drive

`work/` is where every run writes. Its folders point at Drive.

In [ ]:
os.makedirs('work', exist_ok=True)
for name in ['data', 'points', 'models', 'outputs', 'exp_out']:
    target = f'{DRIVE_WORK}/{name}'
    link = f'work/{name}'
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(link):
        if os.path.exists(link):
            raise RuntimeError(f'{link} is a real folder; move its contents to {target} and delete it')
        os.symlink(target, link)
!ls -l work

## 4. Set up

Downloads the authors' code, applies our patches, and builds GeoSteiner and RMST. This takes a few minutes in a fresh session.

In [ ]:
!bash scripts/setup.sh

## 5. Pretrained model (optional)

The authors' trained model is about 770 MB. It's saved to Drive, so this downloads only once.

In [ ]:
!bash scripts/get_pretrained.sh

In [ ]:
!bash scripts/arora.sh pretrained_solve

## 6. Smoke test

Generates small temporary train/val/test data, then trains for a few epochs. The goal is only to check that everything runs.

In [ ]:
!bash scripts/smoke_test.sh

## 7. Training curves

In [ ]:
%load_ext tensorboard
%tensorboard --logdir work/outputs

## Your own runs

Add a config to `configs/` (copy `smoke.yaml` as a starting point), push it, pull it here with step 2, then run:

```
!bash scripts/arora.sh <config-name> [overrides]
```

For example, `!bash scripts/arora.sh smoke 'flow=[train]' train.epochs=10`. Put quotes around overrides that contain `[ ]`.